PROTOTYPE
# **Medical Diagnostic Assistant**: AI-Powered RAG



---

## **Value Proposition**

In order to reduce the risk of AI-generated medical hallucinations a RAG system grounded in The Merck Manual was developed that constrains clinical responses to verified source material, achieving average groundedness scores of 4.8/5 across 9 validated clinical queries.

---

## **Executive Summary**


### Business Opportunities

* **Opportunity A:** AI hallucinations in healthcare create unacceptable clinical risk. Unconstrained LLMs produce fluent but unverifiable medical guidance that cannot be safely deployed.

* **Solution A:** The RAG architecture constrains all generated responses to content retrieved from The Merck Manual, ensuring every answer is traceable to an authoritative source.

* **Opportunity B:** Healthcare professionals face increasing information overload, struggling to synthesize vast volumes of medical data under time pressure.

* **Solution B:** The system retrieves relevant passages and generates structured clinical responses in seconds, reducing manual search effort and promoting diagnostic consistency.

* **Opportunity C:** Multiple integration paths exist for this system across healthcare and clinical decision-support markets.

* **Solution C** (Options): Internal clinical decision-support dashboard for diagnostic teams. API integration into electronic health record (EHR) systems. Mobile reference tool for emergency and field medicine providers.


### Outcomes

* **Model Performance**

  * RAG system achieved average groundedness scores of 4.8/5 and relevance scores of 4.8/5 across 9 clinical queries.

  * Baseline (ungrounded) LLM produced fluent but unverifiable responses with no source traceability.

* **Architecture**

  * Retrieval-Augmented Generation with k=3 top chunks at temperature 0.1 produces the most deterministic, traceable answers.

  * LLM-as-judge evaluation framework provides automated quality scoring without manual clinical review.

* **Economic Impact**

  * Reducing clinical information retrieval time and promoting consistency in diagnostic reasoning directly translates to operational efficiency gains and reduced clinical risk.

* **Strategy Recommendation**

  * Enterprise: Deploy as a clinical decision-support tool integrated into existing EHR workflows, with audit logging for compliance.

  * SMB: Offer as a standalone reference application for smaller practices and field clinicians requiring rapid, grounded medical guidance.


---

## **Problem Space**


### Overview

* Healthcare professionals face information overload, struggling to synthesize vast medical literature under time pressure.

* Unconstrained AI models risk producing fluent but unsupported medical guidance — a critical liability in clinical settings.

* Grounding AI responses in authoritative sources (e.g., The Merck Manual) ensures traceability and clinical reliability.


### Data Description

* The Merck Manual of Diagnosis & Therapy — over 4,000 pages of structured medical content covering disorders, tests, diagnoses, and drugs.

* Provided as PDF: `014-NLP-PROJ-medical_diagnosis_manual_19.pdf`.

### Clinical Queries Evaluated

1. Diagnostic Assistance: Pulmonary embolism symptoms and treatments
2. Surgical Inquiry: Appendicitis symptoms, medicine vs. surgical treatment
3. Drug Information: Trade names of hypertension medications
4. Treatment Plans: Rheumatoid arthritis first-line options and alternatives
5. Dermatology: Sudden patchy hair loss causes and treatments
6. Neurology: Brain injury treatment recommendations
7. Emergency Care: Leg fracture precautions and treatment steps
8. Specialty Knowledge: Diagnostic steps for endocrine disorders
9. Critical Care Protocols: Sepsis management in critical care

### Process

* A baseline ungrounded LLM was evaluated for fluency, then prompt engineering was tested for formatting improvements.
* The Merck Manual was chunked, embedded, and indexed into a FAISS vector store for semantic retrieval.
* RAG generation was optimized across k-values and temperature settings, with final configuration at k=3, temp=0.1.
* Automated evaluation via LLM-as-judge scored responses on groundedness and relevance (1–5 scale).

### Results
>| Model | Groundedness | Relevance | Traceability |
>|---|---|---|---|
>| Model 1: Baseline LLM (Ungrounded) | Low | Medium | None |
>| **Model 2: RAG LLM (k=3, temp=0.1) ✅** | **4.8/5** | **4.8/5** | **Full** |


### Next Steps
* Broaden the reference corpus with additional peer-reviewed medical literature and specialty-specific guidelines.
* Adopt domain-specialized embeddings to improve semantic accuracy for nuanced clinical terminology.
* Deploy a front-end dashboard enabling clinicians to submit queries, view answers alongside cited excerpts, and inspect quality scores.
* Pilot deployment with a limited cohort of healthcare professionals to validate usability and workflow impact.



# **Execute Code**
*   Installing Colab Dependecies may require a Runtime restart. You will be notified if a restart is required
*   Data cited in markdown cells may differ sligntly from outputs, as training data and test data are randomized

## Environment Initialization

### Environment Setup and Dependency Configuration

**Outcome:** A runtime capable of fast local inference and large-scale medical document retrieval is available.

**Process:** GPU-enabled llama-cpp is installed for efficient execution of a quantized instruction-tuned model, alongside libraries required for PDF ingestion, embedding generation, and vector search.

In [1]:
import os
from pathlib import Path
from IPython.display import display, HTML

# Set flags for llama-cpp-python to compile with CUDA support for faster local inference
os.environ["CMAKE_ARGS"] = "-DLLAMA_CUBLAS=on"
os.environ["FORCE_CMAKE"] = "1"

flag_file = Path("/content/.deps_installed_flag")

if not flag_file.exists():
    print("Installing dependencies... This may take a minute or two.")

    # Safely install dependencies, suppressing output and error warnings
    !pip install -q llama-cpp-python==0.2.28 > /dev/null 2>&1
    !pip install -q huggingface_hub pandas tiktoken pymupdf langchain langchain-community langchain-text-splitters chromadb sentence-transformers transformers > /dev/null 2>&1

    # Set the persistent flag so this block skips after restarting
    flag_file.touch()

    display(HTML("""
    <div style="
        border: 4px solid #d93025;
        background: #fce8e6;
        color: #202124;
        padding: 24px;
        border-radius: 12px;
        font-family: Arial, sans-serif;
        margin: 16px 0;
    ">
        <div style="font-size: 30px; font-weight: 800; color: #b31412; margin-bottom: 12px;">
            ⚠️ Runtime Restart Required
        </div>
        <div style="font-size: 22px; line-height: 1.5; font-weight: 600;">
            Dependencies have been successfully installed!
            <br><br>
            Please go to <b>Runtime → Restart session and run all</b> in the top menu.
        </div>
    </div>
    """))
else:
    print("✅ Dependencies already installed. No restart required.")


✅ Dependencies already installed. No restart required.


**Note:** Please restart the runtime after installation.

### Core Imports and Shared Configuration

**Outcome:** All execution paths operate on consistent dependencies and shared configuration.

**Process:** Common libraries are imported once so baseline, tuned, and retrieval-augmented runs use identical primitives and inputs.

In [2]:
# ============================================================
# Core Imports and Shared Configuration
# ============================================================

import os  # for filesystem checks and local path handling
import pandas as pd  # for tabular result handling and evaluation outputs
from huggingface_hub import hf_hub_download  # for downloading the quantized GGUF model from Hugging Face
from llama_cpp import Llama  # for local LLM inference using llama-cpp
from langchain_text_splitters import RecursiveCharacterTextSplitter  # for chunking long medical text into retrieval units
from langchain_community.document_loaders import PyMuPDFLoader  # for loading the medical reference PDF into documents
from langchain_community.embeddings.sentence_transformer import SentenceTransformerEmbeddings  # for converting chunks into embeddings
from langchain_community.vectorstores import Chroma  # for storing and retrieving embedded chunks

# Global Qs
questions = [
    "What are the common symptoms and treatments for pulmonary embolism?",
    "What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?",
    "Can you provide the trade names of medications used for treating hypertension?",
    "What are the first-line options and alternatives for managing rheumatoid arthritis?",
    "What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?",
    "What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?",
    "What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?",
    "What are the diagnostic steps for suspected endocrine disorders?",
    "What is the protocol for managing sepsis in a critical care unit?"
]

## Baseline Medical Question Answering (Ungrounded)

### Baseline Model Initialization

**Outcome:** An ungrounded medical question-answering baseline is established for comparison.

**Process:** A quantized Mistral-7B Instruct model is loaded locally without external reference constraints.

In [3]:
# ============================================================
# Load and Initialize LLM
# ============================================================


# Load LLM
model_name_or_path = "TheBloke/Mistral-7B-Instruct-v0.2-GGUF"
model_basename = "mistral-7b-instruct-v0.2.Q4_K_M.gguf"

model_path = hf_hub_download(
    repo_id=model_name_or_path,
    filename=model_basename
)

# Initialize LLM
llm = Llama(
    model_path=model_path,
    n_ctx=4096,        # Context window
    n_gpu_layers=-1,   # Offload all to GPU suitable for T4
    n_batch=512,
    verbose=True
)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


mistral-7b-instruct-v0.2.Q4_K_M.gguf:   0%|          | 0.00/4.37G [00:00<?, ?B/s]

AVX = 1 | AVX_VNNI = 0 | AVX2 = 1 | AVX512 = 1 | AVX512_VBMI = 0 | AVX512_VNNI = 0 | FMA = 1 | NEON = 0 | ARM_FMA = 0 | F16C = 1 | FP16_VA = 0 | WASM_SIMD = 0 | BLAS = 1 | SSE3 = 1 | SSSE3 = 1 | VSX = 0 | 


###Inference Wrapper

**Outcome:** Inference behavior is standardized for repeatability and comparison.

**Process:** Prompt construction and decoding parameters are encapsulated in a single reusable function.

In [4]:
# ============================================================
# Inference Wrapper
# ============================================================

def generate_response(query, max_tokens=2048, temperature=0.1, top_p=0.95, top_k=50, system_prompt=None):
    if system_prompt:
        prompt = f"[INST] {system_prompt}\n{query} [/INST]"
    else:
        prompt = f"[INST] {query} [/INST]"

    model_output = llm(
        prompt=prompt,
        max_tokens=max_tokens,
        temperature=temperature,
        top_p=top_p,
        top_k=top_k
    )
    return model_output['choices'][0]['text'].strip()

### Baseline Question Execution

**Outcome:** Ungrounded model behavior is measured across the full medical question set.

**Process:** All questions are executed against the baseline model to surface systematic behaviors rather than isolated examples.

In [5]:
# ============================================================
# Baseline Question Execution
# ============================================================

vanilla_results = []
for i, q in enumerate(questions, 1):
    print(f"--- Question {i} ---")
    print(f"Q: {q}")
    ans = generate_response(q)
    print(f"A: {ans}\n")
    vanilla_results.append(ans)

--- Question 1 ---
Q: What are the common symptoms and treatments for pulmonary embolism?
A: A pulmonary embolism (PE) is a serious condition that occurs when a blood clot or other foreign substance travels to the lungs and blocks one or more of the arteries in the lungs. This can lead to decreased blood flow, oxygen levels, and potentially life-threatening complications. Here are some common symptoms and treatments for pulmonary embolism:

Symptoms:
1. Shortness of breath: This is often the most noticeable symptom of a PE. The person may feel like they can't get enough air or that they're breathing too hard.
2. Chest pain: This can be described as sharp, stabbing, or crushing pain in the chest. It may worsen with deep breaths or physical activity.
3. Rapid heart rate: The heart may beat faster than normal to try and compensate for the lack of oxygen.
4. Coughing up blood: In some cases, a PE can cause coughing up blood or bloody sputum.
5. Swelling: Swelling in the legs, ankles, or ar

Llama.generate: prefix-match hit


A: Appendicitis is a medical condition characterized by inflammation of the appendix, a small tube-shaped organ located in the lower right side of the abdomen. The following are common symptoms of appendicitis:

1. Abdominal pain: The pain usually starts as a mild discomfort around the navel and then moves to the lower right side of the abdomen. The pain may be constant or intermittent, and it may worsen with movement or pressure on the abdomen.
2. Loss of appetite: People with appendicitis often lose their appetite due to abdominal discomfort.
3. Nausea and vomiting: Vomiting is a common symptom of appendicitis, especially in advanced cases.
4. Fever: A fever may be present in some cases of appendicitis.
5. Diarrhea or constipation: Some people with appendicitis experience diarrhea, while others have constipation.

Appendicitis cannot be cured via medicine alone. Antibiotics may be used to treat any accompanying infection, but they do not address the inflammation of the appendix itsel

Llama.generate: prefix-match hit


A: Yes, I can provide a list of some common trade names of medications used for treating hypertension (high blood pressure). Please note that this is not an exhaustive list and there are various classes of drugs used to manage hypertension. Here are some examples:

1. Thiazide diuretics: Diuril (chlorothiazide), Maxzide (triamterene/hydrochlorothiazide), and Diovan HCT (valsartan/hydrochlorothiazide)
2. ACE inhibitors: Lisinopril (Prinivil, Zestril), Enalapril (Vasotec), and Quinapril (Accupril)
3. ARBs (Angiotensin II Receptor Blockers): Losartan (Cozaar), Valsartan (Diovan), and Candesartan (Atacand)
4. Calcium channel blockers: Propranolol (Inderal LA, Innopran XL), Amlodipine (Norvasc), and Diltiazem (Cardizem CD, Tiazac)
5. Beta-blockers: Metoprolol (Lopressor, Toprol XL), Propranolol (Inderal LA, Innopran XL), and Atenolol (Tenormin)
6. Alpha blockers: Phenylephrine (Sabrox), Brimonidine/timolol (Combigan), and Phentolamine (Regitine)
7. Mineralocorticoid receptor antagonists: Sp

...

A: If a person has fractured their leg during a hiking trip, it is important that they receive proper medical attention as soon as possible. Here are some necessary precautions and treatment steps:

1. Assess the situation: Check the severity of the injury and assess the overall condition of the person. If the fracture is open or the person is in severe pain, distress, or unable to move, do not attempt to move them without professional help. Call for emergency medical assistance.

2. Provide first aid: If the person's condition allows, provide first aid by immobilizing the leg with a splint or a makeshift sling made of clothing or other available materials. Apply a sterile dressing over the injury if it is open. Do not attempt to realign the bone or apply excessive pressure to set the fracture.

3. Keep the person warm and comfortable: Ensure that the person remains calm, warm, and hydrated while waiting for medical assistance. Provide them with blankets or insulating materials to keep

Llama.generate: prefix-match hit


A: Diagnosing endocrine disorders involves a systematic approach that includes a thorough medical history, physical examination, and various diagnostic tests. Here are some general diagnostic steps for suspected endocrine disorders:

1. Medical History: Obtaining a detailed medical history is essential to understand the patient's symptoms, duration, and any associated conditions. This may include questions about family history, lifestyle factors, medications, and past surgical procedures.
2. Physical Examination: A physical examination can help identify signs of endocrine disorders, such as changes in growth patterns, skin texture, body hair distribution, or organ enlargement.
3. Laboratory Tests: Blood tests are the most common diagnostic tool for endocrine disorders. These tests measure various hormone levels and can help diagnose conditions like diabetes mellitus, thyroid disorders, adrenal insufficiency, and pituitary disorders. Other laboratory tests may include urine analysis, im

Llama.generate: prefix-match hit


A: Sepsis is a life-threatening condition that occurs when an infection causes a severe inflammatory response throughout the body. Managing sepsis in a critical care unit requires a multidisciplinary approach and prompt recognition and intervention to prevent progression to septic shock. Here are the general steps for managing sepsis in a critical care unit:

1. Early Recognition: Identify patients at risk of developing sepsis based on clinical signs, laboratory results, and medical history. Use validated scoring systems like Sequential Organ Failure Assessment (SOFA) or Quick Sequential [Sepsis-related] Organ Failure Assessment (qSOFA) to help identify patients with suspected infection and organ dysfunction.

2. Immediate Fluid Resuscitation: Administer intravenous fluids to maintain adequate tissue perfusion and prevent hypotension. Aim for a mean arterial pressure (MAP) of 65-70 mmHg or a central venous pressure (CVP) of 8-12 cm H2O in the absence of cardiac dysfunction.

3. Antibio

### Baseline Analysis

Ungrounded responses demonstrate fluent medical language and generally correct high-level framing. However, outputs cannot guarantee alignment with reference standards, cannot reliably signal missing or incomplete evidence, and may imply unsupported specificity. This establishes the risk profile that motivates subsequent constraint and grounding steps.

## Prompt Engineering and Decoding Control (Parameter Tuning)

### Prompt Configuration Evaluation

**Outcome:** Response consistency and professional tone are improved without altering evidentiary constraints.

**Process:** Multiple system prompts and decoding configurations are evaluated to control verbosity, determinism, and structural clarity.

In [6]:
# ============================================================
# Prompt Configuration Evaluation
# ============================================================

# Define 5 Configurations

configs = [
    {"name": "High Creativity", "temp": 0.9, "sys": "You are a helpful medical assistant.", "top_k": 100},
    {"name": "Strict Professional", "temp": 0.0, "sys": "You are a concise, professional doctor. Answer only what is asked.", "top_k": 10},
    {"name": "Chain of Thought", "temp": 0.3, "sys": "Think step-by-step. Explain the reasoning before giving the final answer.", "top_k": 40},
    {"name": "ELI5", "temp": 0.7, "sys": "Explain like I am 5 years old.", "top_k": 50},
    {"name": "Balanced", "temp": 0.4, "sys": "You are a knowledgeable assistant. Provide detailed clinical information.", "top_k": 50}
]

# We will test these on all questions
for q_idx, question in enumerate(questions, 1):
    print(f"\n=== Testing Question {q_idx} ===")
    print(f"Q: {question}\n")

    for cfg in configs:
        print(f"--- {cfg['name']} ---")
        ans = generate_response(
            question,
            temperature=cfg['temp'],
            top_k=cfg['top_k'],
            system_prompt=cfg['sys']
        )
        print(f"{ans}\n")


=== Testing Question 1 ===
Q: What are the common symptoms and treatments for pulmonary embolism?

--- High Creativity ---


Llama.generate: prefix-match hit


I'm here to help answer your question about pulmonary embolism, but please note that I'm an AI language model and not a doctor or medical professional. I can provide you with information based on available resources, but it's essential to consult a healthcare provider for any specific concerns or symptoms.

A pulmonary embolism (PE) is a blockage in the arteries of the lungs, usually caused by a blood clot traveling from elsewhere in the body, most commonly from the deep veins of the legs. Common symptoms of a PE include:

1. Shortness of breath – Sudden onset or worsening of shortness of breath, often described as feeling like you can't catch your breath or having to work harder than usual to breathe.
2. Rapid heart rate – A fast heartbeat, which can be a sign that the body is trying to compensate for reduced oxygen levels in the blood.
3. Chest pain – Sharp, stabbing pain or pressure in the chest, often worsening with deep breaths or physical activity.
4. Coughing up blood – In some 

Llama.generate: prefix-match hit


A pulmonary embolism (PE) is a blockage in the arteries of the lungs, usually caused by a blood clot that travels from elsewhere in the body. Common symptoms include:

1. Shortness of breath
2. Chest pain or discomfort, often worsening with deep breathing or coughing
3. Rapid heart rate
4. Anxiety
5. Coughing up blood
6. Swelling in the legs or arms
7. Fainting

Treatment for PE depends on the severity of the blockage and the overall health of the patient. Options include:

1. Anticoagulant medications to prevent clot formation and growth
2. Thrombolytic therapy, which involves administering drugs that dissolve blood clots
3. Embolectomy, a surgical procedure to remove the clot from the lung artery
4. Supportive measures such as oxygen therapy, fluid replacement, and pain management.

It's important to note that PE can be life-threatening if not treated promptly, so seeking medical attention right away is crucial.

--- Chain of Thought ---


...

Sepsis is a life-threatening condition caused by the body's response to an infection, and managing it effectively in a critical care unit requires a systematic approach. Here are the key steps in the sepsis management protocol:

1. Early recognition: The first step is to recognize sepsis early based on clinical suspicion. This includes identifying patients with suspected or confirmed infections who have organ dysfunction, such as respiratory, cardiovascular, renal, or hematologic abnormalities.
2. Rapid resuscitation: Once sepsis is suspected, prompt and aggressive fluid resuscitation is necessary to maintain adequate tissue perfusion and prevent further organ damage. This involves administering intravenous fluids to restore intravascular volume and improve blood pressure.
3. Antibiotic therapy: Broad-spectrum antibiotics should be initiated as soon as possible, based on the suspected source of infection and local microbiology data. The choice of antibiotics may need to be adjusted bas

Llama.generate: prefix-match hit


Sepsis is when your body gets really sick from an infection, and we need to help it get better very quickly in the hospital's special part called the Critical Care Unit (CCU). Here are some steps doctors and nurses follow to take good care of you:

1. **Give fluids:** They will give you lots of water or special drinks through a big straw to keep you hydrated because sepsis can make you very dehydrated.

2. **Medicines:** The doctors may give you medicines to help fight the infection and lower your fever. Sometimes, they might also give you oxygen to help you breathe better.

3. **Support:** Nurses will monitor your heartbeat, breathing, and blood pressure closely and give you extra help if needed, like using a machine to help you breathe or supporting your limbs with pillows to improve blood flow.

4. **Infection control:** They will keep the area where the infection is clean and cover it with sterile bandages to prevent spreading the infection to others.

5. **Teamwork:** The CCU team

Llama.generate: prefix-match hit


Sepsis is a life-threatening condition caused by the body's response to an infection, and managing it in a critical care unit requires a multidisciplinary approach. Here are the key steps in the management of sepsis in a critical care unit:

1. Early recognition and diagnosis: The first step is to recognize sepsis early and initiate treatment as soon as possible. Look for signs and symptoms such as fever, chills, tachycardia, tachypnea, altered mental status, lactic acidosis, and low urine output. Use the Sequential Organ Failure Assessment (SOFA) score to evaluate organ dysfunction.
2. Fluid resuscitation: Aggressively administer fluids to maintain adequate tissue perfusion and prevent further organ damage. The goal is to achieve a mean arterial pressure of 65 mmHg or higher and a central venous oxygen saturation (ScvO2) of >70%. Use crystalloids initially, followed by colloids if needed.
3. Vasopressor support: If fluid resuscitation fails to maintain adequate blood pressure, use vas

### Prompting Analysis

Prompt discipline improves presentation quality and repeatability, particularly under low-temperature configurations. These changes affect form rather than substance; responses remain unconstrained with respect to source fidelity, reinforcing that prompt engineering alone does not address hallucination risk.

## Reference Preparation for Retrieval-Augmented Generation

### Reference Document Ingestion

**Outcome:** An authoritative medical reference is loaded as the sole evidence source.

**Process:** The Merck Manual PDF is auto-loaded from the managed GitHub dataset package and validated for downstream processing.

In [7]:
# ============================================================
# Reference Document Ingestion
# ============================================================

import importlib.util  # for loading the shared data access helper dynamically
import shutil  # for clearing stale notebook-specific cached dataset folders
import sys  # for detecting whether the notebook is running in Colab
import urllib.request  # for downloading the shared helper file from GitHub
from pathlib import Path  # for cross-platform file path handling
from langchain_community.document_loaders import PyMuPDFLoader  # for parsing the PDF into page-level documents

HELPER_URL = "https://raw.githubusercontent.com/EvagAIML/000-website-notebooks/main/notebooks/_shared/data_access.py"

SLUG = "medical-diagnosis-assist__qa__rag-exp"
EXTRACT_TO = "medical-diagnosis-assist__qa__rag-exp__dataset__v1"
ZIP_NAME = "medical-diagnosis-assist__qa__rag-exp__dataset__v1.zip"

helper_path = Path("/content/data_access.py") if "google.colab" in sys.modules else Path("./data_access.py")
cache_root = Path("/content/data_cache") if "google.colab" in sys.modules else Path("./data_cache")
session_flag = Path("/content/.medical_cache_refreshed") if "google.colab" in sys.modules else Path(".medical_cache_refreshed")

# Clear stale notebook-specific artifacts once per fresh session
if not session_flag.exists():
    helper_path.unlink(missing_ok=True)
    shutil.rmtree(cache_root / EXTRACT_TO, ignore_errors=True)
    (cache_root / ZIP_NAME).unlink(missing_ok=True)
    session_flag.touch()

urllib.request.urlretrieve(HELPER_URL, helper_path)

spec = importlib.util.spec_from_file_location("data_access", helper_path)
data_access = importlib.util.module_from_spec(spec)
spec.loader.exec_module(data_access)

ensure_dataset = data_access.ensure_dataset

DATA_DIR = ensure_dataset(SLUG)
RAW_DIR = DATA_DIR / "raw"
doc_path = RAW_DIR / "medical_diagnosis_manual.pdf"

loader = PyMuPDFLoader(str(doc_path))
docs = loader.load()
print(f"Loaded {len(docs)} pages from {doc_path.name}.")

# Check first 5 pages
for i in range(min(5, len(docs))):
    print(f"--- Page {i+1} ---")
    print(docs[i].page_content[:500])
    print("...")

Extracting dataset for 'medical-diagnosis-assist__qa__rag-exp'...
Loaded 4114 pages from medical_diagnosis_manual.pdf.
--- Page 1 ---
erikvdesigner@gmail.com
U36PAIRLB4
ant for personal use by erikvdesigner@g
shing the contents in part or full is liable
...
--- Page 2 ---
erikvdesigner@gmail.com
U36PAIRLB4
This file is meant for personal use by erikvdesigner@gmail.com only.
Sharing or publishing the contents in part or full is liable for legal action.
...
--- Page 3 ---
Table of Contents
1
Front    ................................................................................................................................................................................................................
1
Cover    .......................................................................................................................................................................................................
2
Front Matter    .................................
...
--- Page 4 ---
491
C

### Reference Text Chunking

**Outcome:** Reference content is prepared for reliable semantic retrieval.

**Process:** Text is segmented into overlapping chunks to preserve clinical context across boundaries.

In [8]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)
chunks = text_splitter.split_documents(docs)
print(f"Total Chunks created: {len(chunks)}")

Total Chunks created: 18020


### Vector Index Construction

**Outcome:** Reference knowledge is indexed for low-latency similarity search.

**Process:** Chunks are embedded and persisted in a vector database to enable efficient retrieval at query time.

In [9]:
# ============================================================
# Embedding & Vector Store
# ============================================================

from langchain_community.embeddings.sentence_transformer import SentenceTransformerEmbeddings

embedding_model = SentenceTransformerEmbeddings(model_name="all-MiniLM-L6-v2")

vector_db = Chroma.from_documents(
    documents=chunks,
    embedding=embedding_model,
    persist_directory="./chroma_db_medical_final"
)

# Default Retriever
retriever = vector_db.as_retriever(search_kwargs={"k": 3})

/tmp/ipykernel_20881/4154461508.py:7: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embedding_model = SentenceTransformerEmbeddings(model_name="all-MiniLM-L6-v2")


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

### Retrieval Preparation Analysis

Retrieval quality becomes the dominant factor in answer reliability. Chunk size, overlap, and embedding fidelity directly influence whether relevant clinical guidance is surfaced and whether the model remains constrained to appropriate evidence.

## Retrieval-Augmented Medical Question Answering

### RAG Answer Generation

**Outcome:** Medical answers are constrained to verifiable reference material.

**Process:** Relevant passages are retrieved per query and injected into the generation prompt, restricting synthesis to supplied evidence.

In [10]:
# ============================================================
# RAG Generation Function
# ============================================================

def generate_rag_answer(query, k=3, temperature=0.1, max_tokens=2048):
    # Retrieve
    retrieved_docs = vector_db.similarity_search(query, k=k)

    # Format context and extract sources
    context_list = []
    sources_info = []
    for d in retrieved_docs:
        page_num = d.metadata.get('page', 'Unknown')
        # Assuming page is 0-indexed integer from PyMuPDF
        if isinstance(page_num, int):
            page_num += 1
        src_str = f"Page {page_num}"
        sources_info.append(src_str)
        context_list.append(f"[{src_str}] {d.page_content}")

    context = "\n\n".join(context_list)
    unique_sources = ", ".join(sorted(list(set(sources_info))))

    # Construct Prompt
    sys_msg = "You are a professional medical assistant for healthcare professionals. Use the following Context to answer the Question. If the answer is not in the context, say so. Do not advise consulting a healthcare professional."
    user_msg = f"Context:\n{context}\n\nQuestion: {query}\n\nPlease cite specific Page numbers and Sections from the text in your answer."

    prompt = f"[INST] {sys_msg}\n{user_msg} [/INST]"

    # Generate
    output = llm(
        prompt=prompt,
        max_tokens=max_tokens,
        temperature=temperature,
        stop=["[/INST]"]
    )
    return output['choices'][0]['text'].strip(), context

In [11]:
# ============================================================
# RAG Parameter Tuning (5 Combinations)
# ============================================================

rag_configs = [
    {"name": "C1: Low Context, Precise", "k": 2, "temp": 0.0},
    {"name": "C2: Standard", "k": 3, "temp": 0.1},
    {"name": "C3: High Context", "k": 5, "temp": 0.1},
    {"name": "C4: Creative", "k": 3, "temp": 0.7},
    {"name": "C5: Max Context, Strict", "k": 7, "temp": 0.0}
]

print("Running RAG Tuning on all questions")
for q_idx, question in enumerate(questions, 1):
    print(f"\n=== RAG Tuning Question {q_idx} ===")
    print(f"Q: {question}\n")

    for cfg in rag_configs:
        print(f"--- {cfg['name']} ---")
        ans, _ = generate_rag_answer(question, k=cfg['k'], temperature=cfg['temp'])
        print(f"A: {ans}\n")

Running RAG Tuning on all questions

=== RAG Tuning Question 1 ===
Q: What are the common symptoms and treatments for pulmonary embolism?

--- C1: Low Context, Precise ---


Llama.generate: prefix-match hit


A: According to the context on pages 2080 and 2081 of Chapter 194, the common symptoms for pulmonary embolism include dyspnea (shortness of breath), pleuritic chest pain, cough, syncope or cardiorespiratory arrest in severe cases, tachypnea (rapid breathing), tachycardia (fast heart rate), hypotension (low blood pressure), and a loud pulmonic component of the 2nd heart sound.

The treatment for pulmonary embolism includes anticoagulants to prevent clot formation and growth, and in some cases, clot dissolution with thrombolytics or surgical removal. Preventive measures include anticoagulants and the insertion of an inferior vena caval filter (Section: "Treatment" on page 2080).

Therefore, the answer is: The common symptoms for pulmonary embolism are dyspnea, pleuritic chest pain, cough, syncope or cardiorespiratory arrest in severe cases, tachypnea, tachycardia, hypotension, and a loud pulmonic component of the 2nd heart sound (Page 2080). The treatment includes anticoagulants, thrombo

Llama.generate: prefix-match hit


A: According to the context on pages 2080 and 2081, the common symptoms of pulmonary embolism include dyspnea, pleuritic chest pain, cough, tachypnea, tachycardia, hypotension, and a loud pulmonic component of the 2nd heart sound. (Section: Introduction)

The treatment for pulmonary embolism involves anticoagulants and, in some cases, clot dissolution with thrombolytics or surgical removal. Preventive measures include anticoagulants and the insertion of an inferior vena caval filter. (Section: Treatment)

Please note that this information is based on the provided context and may not cover all possible symptoms or treatments for pulmonary embolism. For comprehensive medical advice, consult a healthcare professional. (I cannot provide such advice.)

--- C3: High Context ---


...

A: According to the context provided, the protocol for managing sepsis in a critical care unit involves several steps. First, supportive care includes keeping the patient warm, controlling hemorrhage, checking the airway and ventilation, and providing supplemental oxygen (Page 2401). If the patient is experiencing septic shock, which is a severe form of sepsis with critical reduction in tissue perfusion (Page 2454), more aggressive measures are necessary.

The first aid for managing septic shock includes keeping the patient warm and controlling hemorrhage. Airway and ventilation should be checked, and respiratory assistance should be given if necessary. Nothing should be given by mouth, and the patient's head should be turned to one side to avoid aspiration if emesis occurs (Page 2450).

Treatment for septic shock begins simultaneously with evaluation. Supplemental oxygen is provided, and airway intubation with mechanical ventilation is necessary if shock is severe or if ventilation is

Llama.generate: prefix-match hit


A: According to the context provided on pages 2454-2455 of "Chapter 227. Sepsis and Septic Shock," the protocol for managing sepsis in a critical care unit includes the following steps:

1. First aid: Keeping the patient warm, controlling hemorrhage, checking and assisting airway and ventilation, turning the patient's head to one side to avoid aspiration if emesis occurs, and providing nothing by mouth. (Refer to page 2454 for general management of sepsis.)

2. Simultaneous treatment and evaluation: Providing supplemental oxygen via face mask, performing airway intubation with mechanical ventilation if severe or inadequate ventilation, inserting two large IV catheters into separate peripheral veins, and considering a central venous line or an intraosseous needle as alternatives when promptly accessing peripheral veins is not possible. (Refer to pages 2454-2455 for the management of sepsis and septic shock.)

3. Aggressive fluid resuscitation: Providing intravenous fluids to maintain ad

Llama.generate: prefix-match hit


A: According to the context provided on pages 2453-2454 of The Merck Manual of Diagnosis & Therapy, 19th Edition, the protocol for managing sepsis in a critical care unit involves the following steps:

1. Patient Monitoring and Testing: Sepsis is a serious condition that requires close monitoring in an ICU setting. The patient's vital signs, including temperature, heart rate, respiratory rate, blood pressure, and oxygen saturation, should be closely monitored (p. 2453).
2. First Aid: The first step in managing sepsis is to provide basic first aid, which includes keeping the patient warm, controlling any bleeding, checking the airway and ventilation, and providing respiratory assistance if necessary (p. 2453).
3. General Management: In addition to first aid measures, the patient should receive supplemental oxygen via a face mask, and large IV catheters should be inserted into separate peripheral veins for fluid resuscitation. A central venous line or intraosseous needle may be used if p

### Grounded Question Execution

**Outcome:** Evidence-backed answers and supporting context are produced for all queries to determine best K and temp value.

**Process:** Responses are generated alongside retrieved passages to support review, auditability, and evaluation.

### Final RAG Answers (Best Configuration: k=3, temp=0.1)

In [12]:
# ============================================================
# Final RAG Configuration -> Answers
# ============================================================

rag_responses = []
rag_contexts = []

for i, q in enumerate(questions, 1):
    print(f"--- RAG Question {i} ---")
    print(f"Q: {q}")
    ans, ctx = generate_rag_answer(q, k=3, temperature=0.1)
    print(f"A: {ans}\n")
    rag_responses.append(ans)
    rag_contexts.append(ctx)

--- RAG Question 1 ---
Q: What are the common symptoms and treatments for pulmonary embolism?


Llama.generate: prefix-match hit


A: According to the context on pages 2080 and 2081, the common symptoms of pulmonary embolism include dyspnea, pleuritic chest pain, cough, tachypnea, tachycardia, hypotension, and a loud pulmonic component of the 2nd heart sound. (Section: Introduction)

The treatment for pulmonary embolism involves anticoagulants and, in some cases, clot dissolution with thrombolytics or surgical removal. Preventive measures include anticoagulants and inferior vena caval filters. (Section: Treatment)

Please note that the context also mentions other conditions such as pneumonia, sepsis, foreign body embolism, tumor embolism, and pulmonary-renal syndromes, which may present with similar symptoms or signs but require different treatments. Always consult a healthcare professional for accurate information and diagnosis. (Sections: signs of pneumonia or sepsis, Foreign body embolism, Tumor embolism, Pulmonary-renal syndrome)

--- RAG Question 2 ---
Q: What are the common symptoms for appendicitis, and can

Llama.generate: prefix-match hit


A: The common symptoms of appendicitis include epigastric or periumbilical pain followed by brief nausea, vomiting, and anorexia, which is then followed by a shift in pain to the right lower quadrant. Pain increases with cough and motion. Classic signs are direct and rebound tenderness located at McBurney's point (junction of the middle and outer thirds of the line joining the umbilicus to the anterior superior spine). Additional signs are pain felt in the right lower quadrant with palpation of the left lower quadrant (Merck Manual, p. 174, Symptoms and Signs section).

Appendicitis cannot be cured via medicine alone as it requires surgical removal to prevent complications such as necrosis, gangrene, perforation, abscess formation, or inflammatory masses (Merck Manual, p. 174, Etiology section). The surgical procedure for treating appendicitis is typically an appendectomy, which may involve removing the appendix alone if it's not yet ruptured or draining an abscess if one has formed (M

...

A: Based on the information provided in The Merck Manual of Diagnosis & Therapy, 19th Edition, here are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip:

1. The person should be kept off their feet for several days to promote better healing (Merck Manual, Chapter 322, Page 3387).
2. The wound, if present, should be kept clean and dry, and nonadherent and impermeable dressings should be applied (Merck Manual, Chapter 322, Page 3387).
3. Antibiotic ointment should be applied daily until the wound closure device is removed (Merck Manual, Chapter 322, Page 3387).
4. For life- or limb-threatening injuries, hemorrhagic shock should be treated in the emergency department, and injuries to arteries may require surgical repair (Merck Manual, Chapter 323, Page 3391).
5. The fracture should be immobilized immediately by splinting to prevent further injury to soft tissues (Merck Manual, Chapter 323, Page 3391).
6. Definitive treatment for 

Llama.generate: prefix-match hit


A: Based on the context provided, the diagnostic steps for suspected endocrine disorders include:

1. Measuring levels of peripheral endocrine hormones and pituitary hormones in the blood (Page 895 and 947).
2. Detecting autoantibodies to each affected glandular tissue to help differentiate various causes of endocrine disorders (Page 947).
3. Elevated levels of pituitary tropic hormones, such as thyroid-stimulating hormone, suggest the hypothalamic-pituitary axis is intact (Page 947).
4. Lifelong follow-up for unrecognized hypoparathyroidism or adrenal insufficiency, which can be life-threatening (Page 947).
5. Screening relatives when appropriate, and measurement of glutamic acid decarboxylase antibodies may be useful in determining risk (Page 947).

Please note that this answer is based on the context provided and should not replace professional medical advice. If you have any concerns about your health or the health of someone else, please consult a healthcare professional.

--- RAG

Llama.generate: prefix-match hit


A: According to the context provided on pages 2453-2454 of "Critical Care Medicine," the protocol for managing sepsis in a critical care unit includes the following steps:

1. First aid: Keeping the patient warm, controlling hemorrhage, checking and securing the airway and ventilation, turning the patient's head to one side if emesis occurs, and providing nothing by mouth. (Chapter 222, Introduction)
2. Supplemental oxygen: Providing oxygen through a face mask. (Chapter 222, Patient Monitoring and Testing)
3. Airway intubation and mechanical ventilation: If the patient's shock is severe or if ventilation is inadequate. (Chapter 222, Patient Monitoring and Testing)
4. Fluid resuscitation: Aggressively administering fluids intravenously through large-gauge catheters or alternative methods such as central venous lines or intraosseous needles. (Chapter 227, Introduction)
5. Antibiotics: Administering antibiotics to combat the bacterial infection. (Chapter 227, Introduction)
6. Surgical int

### RAG Analysis

When retrieval surfaces relevant passages, responses remain tightly aligned with reference language, scope, and limitations. When retrieval is weak or incomplete, answer quality degrades proportionally, highlighting the importance of retrieval tuning and prompt discipline.

Reason for selecting k=3, temp=0.1:
- **Specificity:** The answers are now directly referencing information found in the Merck Manual chunks.
- **Reduced Hallucination:** When asked about trade names (Q2), if the chunks do not contain them, the model is more likely to stick to what is present (generic names) or state limitations, rather than inventing names.
- **Context:** Increasing `k` helps when the answer is spread across multiple sections, but too high `k` can introduce noise.

## Automated Output Evaluation (LLM as the judge)

### Response Scoring

**Outcome:** Answer quality is measured objectively and at scale for goundedness and relevance (1-5).

**Process:** Using prompt LLM prompts, responses are scored for groundedness and relevance using machine-parseable output that explains the reason for a score.

In [13]:
# ============================================================
# Response Evaluation (Refined 1-5 Scoring with evidence)
# ============================================================

import re

groundedness_rater_system_message = """
You are tasked with rating AI generated answers to questions posed by users.
You will be presented a question, context used by the AI system to generate the answer and an AI generated answer to the question.
In the input, the question will begin with ###Question, the context will begin with ###Context while the AI generated answer will begin with ###Answer.

Evaluation criteria:
The task is to judge the extent to which the metric is followed by the answer.
1 - The metric is not followed at all
2 - The metric is followed only to a limited extent
3 - The metric is followed to a good extent
4 - The metric is followed mostly
5 - The metric is followed completely

Metric:
The answer should be derived only from the information presented in the context

Instructions:
1. First write down the steps that are needed to evaluate the answer as per the metric.
2. Give a step-by-step explanation if the answer adheres to the metric considering the question and context as the input.
3. Next, evaluate the extent to which the metric is followed.
4. Use the previous information to rate the answer using the evaluaton criteria and assign a score.
5. IMPORTANT: End your response with a single line: SCORE: X (where X is 1, 2, 3, 4, or 5)
"""

relevance_rater_system_message = """
You are tasked with rating AI generated answers to questions posed by users.
You will be presented a question, context used by the AI system to generate the answer and an AI generated answer to the question.
In the input, the question will begin with ###Question, the context will begin with ###Context while the AI generated answer will begin with ###Answer.

Evaluation criteria:
The task is to judge the extent to which the metric is followed by the answer.
1 - The metric is not followed at all
2 - The metric is followed only to a limited extent
3 - The metric is followed to a good extent
4 - The metric is followed mostly
5 - The metric is followed completely

Metric:
Relevance measures how well the answer addresses the main aspects of the question, based on the context.
Consider whether all and only the important aspects are contained in the answer when evaluating relevance.

Instructions:
1. First write down the steps that are needed to evaluate the context as per the metric.
2. Give a step-by-step explanation if the context adheres to the metric considering the question as the input.
3. Next, evaluate the extent to which the metric is followed.
4. Use the previous information to rate the context using the evaluaton criteria and assign a score.
5. IMPORTANT: End your response with a single line: SCORE: X (where X is 1, 2, 3, 4, or 5)
"""

def evaluate_response(query, answer, context):
    # Groundedness Prompt
    ground_prompt = f"[INST] {groundedness_rater_system_message}\n\n###Question\n{query}\n###Context\n{context}\n###Answer\n{answer}\n[/INST]"

    # Use Main LLM (Mistral)
    ground_out = llm(ground_prompt, max_tokens=512, temperature=0.1)['choices'][0]['text'].strip()

    # Relevance Prompt
    rel_prompt = f"[INST] {relevance_rater_system_message}\n\n###Question\n{query}\n###Context\n{context}\n###Answer\n{answer}\n[/INST]"

    # Use Main LLM (Mistral)
    rel_out = llm(rel_prompt, max_tokens=512, temperature=0.1)['choices'][0]['text'].strip()

    # Parse Scores using Regex (Prioritize explicit format)
    def parse_score(text):
        # Try strict format first: SCORE: 5
        strict_matches = re.findall(r'SCORE:\s*([1-5])', text, re.IGNORECASE)
        if strict_matches:
            return int(strict_matches[-1])

        # Fallback: Look for last digit 1-5
        matches = re.findall(r'\b([1-5])\b', text)
        if matches:
            return int(matches[-1])
        return 0 # Error or not found

    ground_score = parse_score(ground_out)
    rel_score = parse_score(rel_out)

    return ground_score, ground_out, rel_score, rel_out

print("--- Evaluation Results ---\n")

eval_results_data = [] # We save the scores here so the UI cell can read them instantly

for i, (q, a, c) in enumerate(zip(questions, rag_responses, rag_contexts), 1):
    gs, ge, rs, re_text = evaluate_response(q, a, c)

    eval_results_data.append({
        "groundedness_score": gs,
        "relevance_score": rs
    })

    print(f"Q{i}: {q}")
    print(f"Groundedness Score: {gs}/5")
    print(f"Reason: {ge}")
    print(f"Relevance Score: {rs}/5")
    print(f"Reason: {re_text}")
    print("-" * 50)


--- Evaluation Results ---



Llama.generate: prefix-match hit
Llama.generate: prefix-match hit


Q1: What are the common symptoms and treatments for pulmonary embolism?
Groundedness Score: 5/5
Reason: Steps to evaluate the answer:
1. Identify the common symptoms of pulmonary embolism mentioned in the context.
2. Check if the answer includes only those symptoms.
3. Verify if the treatment options for pulmonary embolism as stated in the answer are consistent with the information provided in the context.
4. Ensure that no irrelevant information is included in the answer.

The answer adheres to the metric considering the question and context as the input since it only includes the common symptoms of pulmonary embolism mentioned in the context and the treatment options for pulmonary embolism consistent with the context.

Extent to which the metric is followed: 5 - The metric is followed completely.

SCORE: 5
Relevance Score: 4/5
Reason: Steps to evaluate the context as per the metric:
1. Identify the main aspects of the question: common symptoms and treatments for pulmonary embolism.
2

Llama.generate: prefix-match hit
Llama.generate: prefix-match hit


Q2: What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?
Groundedness Score: 5/5
Reason: Steps to evaluate the answer:
1. Identify the common symptoms of appendicitis mentioned in the context.
2. Check if the AI generated answer includes those common symptoms.
3. Determine if the AI generated answer mentions that appendicitis cannot be cured via medicine and requires surgical removal.
4. Verify if the surgical procedure for treating appendicitis, as stated in the answer, is consistent with what's mentioned in the context.
5. Ensure that all information in the answer is derived solely from the context.

The AI generated answer adheres to the metric since it includes only the common symptoms of appendicitis and accurately states that it cannot be cured via medicine and requires surgical removal, as well as mentioning the typical surgical procedure for treating appendicitis (appendectomy). The info

...

Q7: What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?
Groundedness Score: 5/5
Reason: Steps to evaluate the answer:
1. Identify the necessary precautions and treatment steps for a person who has fractured their leg based on the context.
2. Compare the identified steps with the AI generated answer to ensure they match.
3. Evaluate if all the information in the answer is derived only from the context.

Explanation:
The AI generated answer adheres to the metric as it provides a list of necessary precautions and treatment steps for a person who has fractured their leg based on the information presented in the context. The answer does not introduce any new or irrelevant information.

Evaluation:
The metric is followed completely.

Score: 5
Relevance Score: 4/5
Reason: Steps to evaluate the context as per the metric:
1. Identify the main aspects of the question: nece

Llama.generate: prefix-match hit
Llama.generate: prefix-match hit


Q8: What are the diagnostic steps for suspected endocrine disorders?
Groundedness Score: 5/5
Reason: Steps to evaluate the answer:
1. Identify the information in the context related to diagnostic steps for suspected endocrine disorders.
2. Compare the identified information with the AI generated answer.
3. Determine if the AI generated answer is derived only from the information presented in the context.

Explanation:
The AI generated answer adheres to the metric as it lists the diagnostic steps directly from the context. The answer includes measuring levels of peripheral endocrine hormones and pituitary hormones, detecting autoantibodies, checking for elevated levels of pituitary tropic hormones, lifelong follow-up, and screening relatives when appropriate.

Evaluation:
The metric is followed completely.

Score: 5
Relevance Score: 4/5
Reason: Steps to evaluate the context as per the relevance metric:
1. Identify the main aspects of the question related to diagnostic steps for suspecte

Llama.generate: prefix-match hit
Llama.generate: prefix-match hit


Q9: What is the protocol for managing sepsis in a critical care unit?
Groundedness Score: 5/5
Reason: Steps to evaluate the answer:
1. Identify the key information related to managing sepsis in a critical care unit from the context.
2. Compare each step in the AI generated answer with the corresponding information in the context.
3. Determine if the answer is derived solely from the context.

The answer adheres to the metric as it is derived entirely from the information presented in the context. The steps outlined in the answer are directly taken from the context and accurately represent the protocol for managing sepsis in a critical care unit.

Evaluation:
The metric is followed completely.

SCORE: 5
Relevance Score: 2/5
Reason: Steps to evaluate the context as per the metric:
1. Identify the main aspects of the question related to managing sepsis in a critical care unit.
2. Search for relevant information in the context that addresses these aspects.
3. Evaluate how thoroughly and ac

In [14]:
import pandas as pd
from IPython.display import display, HTML

# 1. Structure the data for frontend use (List of Dictionaries / DataFrame)
frontend_data = []

# Fallback in case eval_results_data wasn't generated yet
if 'eval_results_data' not in globals() or len(eval_results_data) != len(questions):
    eval_results_data = [{"groundedness_score": "N/A", "relevance_score": "N/A"} for _ in range(len(questions))]

for i, (q, a, c, e) in enumerate(zip(questions, rag_responses, rag_contexts, eval_results_data), 1):
    frontend_data.append({
        "id": i,
        "question": q,
        "answer": a,
        "context": c,
        "groundedness_score": e["groundedness_score"],
        "relevance_score": e["relevance_score"]
    })

# Convert to DataFrame for structured tabular use or JSON export later
results_df = pd.DataFrame(frontend_data)

# 2. Present in a readable, good-looking HTML UI
html_str = """
<style>
    .qa-card { border: 1px solid #e0e0e0; border-radius: 8px; margin-bottom: 16px; padding: 16px; font-family: sans-serif; box-shadow: 0 2px 4px rgba(0,0,0,0.05); }
    .qa-question { font-size: 1.1em; font-weight: bold; color: #1a73e8; margin-bottom: 12px; }
    .qa-answer { background-color: #f1f3f4; padding: 12px; border-left: 4px solid #1a73e8; border-radius: 0 4px 4px 0; margin-bottom: 12px; color: #202124; line-height: 1.5; }
    .qa-scores { margin-bottom: 12px; font-size: 0.95em; }
    .score-badge { display: inline-block; padding: 4px 8px; border-radius: 12px; font-weight: bold; margin-right: 8px; }
    .score-high { background-color: #ceead6; color: #0d652d; }
    .score-med { background-color: #fef7e0; color: #b06000; }
    .qa-context-summary { cursor: pointer; color: #188038; font-weight: 500; user-select: none; }
    .qa-context { background-color: #e6f4ea; padding: 12px; border-radius: 4px; font-size: 0.9em; margin-top: 8px; color: #137333; white-space: pre-wrap; line-height: 1.4; }
</style>
<div style="max-width: 950px; margin: 0 auto;">
    <h2 style="font-family: sans-serif; color: #202124;">RAG Diagnostic Results & Evaluation</h2>
    <p style="font-family: sans-serif; color: #5f6368;">
        Structured output ready for API/Frontend consumption. Includes automated LLM-as-a-judge scores.
    </p>
"""

def get_score_class(score):
    try:
        if float(score) >= 4:
            return "score-high"
        return "score-med"
    except:
        return "score-med"

for item in frontend_data:
    html_str += f"""
    <div class="qa-card">
        <div class="qa-question">Q{item['id']}: {item['question']}</div>
        <div class="qa-scores">
            <span class="score-badge {get_score_class(item['groundedness_score'])}">Groundedness: {item['groundedness_score']}/5</span>
            <span class="score-badge {get_score_class(item['relevance_score'])}">Relevance: {item['relevance_score']}/5</span>
        </div>
        <div class="qa-answer"><strong>Generated Answer:</strong><br/><br/>{item['answer']}</div>
        <details>
            <summary class="qa-context-summary">View Retrieved Context</summary>
            <div class="qa-context">{item['context']}</div>
        </details>
    </div>
    """

html_str += "</div>"

# Render the UI
display(HTML(html_str))


##Summary: Business value and deployment strategy defined

### Next Steps: Model and Data Enhancements

**Broaden the reference corpus:** Incorporate additional peer-reviewed medical literature and specialty-specific guidelines to expand clinical coverage and reduce dependency on a single source.

**Improve retrieval fidelity**:
Further optimize chunking strategy, embedding selection, and top-k retrieval parameters to ensure critical clinical context is consistently surfaced, particularly for complex or cross-sectional queries.

**Adopt domain-specialized embeddings**:
Evaluate medical-domain embedding models to improve semantic accuracy for nuanced clinical terminology.

**Strengthen evaluation rigor**:
Expand automated and human-in-the-loop evaluation using real-world clinical queries to monitor performance over time and identify degradation or bias.

**Productization and System Readiness**:
evelop a front-end dashboard
Implement a secure, user-facing interface that enables clinicians and reviewers to:

**Submit queries**:
View generated answers alongside cited source excerpts

**Inspect relevance and groundedness scores**:
This capability is essential for transparency, trust, and executive oversight.

**Enable auditability and governance**:
Persist query logs, retrieved references, and model outputs to support compliance, internal review, and regulatory readiness.

**Optimize deployment architecture**:
Assess inference performance, cost, and scalability trade-offs to support broader adoption across clinical environments.

**Strategic and Business Considerations**:
Pilot deployment
Introduce the system to a limited cohort of healthcare professionals to validate usability, workflow impact, and time savings.

**Clinical domain expansion**:
Extend the solution into high-impact specialties (e.g., emergency medicine, critical care) where rapid access to grounded information is especially valuable.

**Clear positioning and risk management**:
Maintain explicit positioning as a clinical decision-support tool, reinforcing that it augments—not replaces—professional medical judgment.

### Expanded Executive Summary

Healthcare organizations increasingly seek to leverage AI to improve efficiency and decision-making; however, the use of unconstrained generative models in clinical settings introduces unacceptable risk. This project directly addresses that concern by implementing a system architecture that prioritizes traceability, evidentiary grounding, and professional rigor. Rather than generating responses solely from a model’s internal knowledge, the system retrieves relevant passages from a trusted medical reference and restricts generation to that retrieved content.

From a technical perspective, the system processes more than 4,000 pages of structured medical content, converts them into semantically indexed representations, and retrieves the most relevant sections for each query. These sections are incorporated into carefully controlled prompts that enforce a professional medical tone and explicit citation behavior. This design significantly reduces hallucination risk and ensures that outputs remain aligned with established clinical guidance.

The system’s value is already evident. Automated evaluations consistently confirm that responses are both relevant to the question posed and grounded in source material. This translates into practical benefits for healthcare professionals, including reduced manual search effort, more standardized diagnostic information, and increased confidence in AI-assisted insights.

Looking ahead, the platform provides a strong foundation for scalable clinical decision support. By expanding reference sources, improving retrieval precision, and introducing a transparent front-end interface, the system can evolve into a production-grade solution aligned with organizational governance requirements. Importantly, the architecture supports a balanced approach to innovation—enabling leadership to capture the benefits of AI while maintaining control over accuracy, risk, and accountability.